In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
TABLES_DIR = os.path.join(OUTPUTS_DIR, "tables")
FIGURES_DIR = os.path.join(OUTPUTS_DIR, "figures")

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

FAKE_LABEL = 1
REAL_LABEL = 0

train_df = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "train.csv"), usecols=["full_text", "label"])
test_df = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "test.csv"), usecols=["full_text", "label"])

X_train = train_df["full_text"].fillna("").astype(str).tolist()
y_train = train_df["label"].astype(int).to_numpy()

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)

model = LinearSVC(class_weight="balanced")
model.fit(X_train_tfidf, y_train)

feat = vectorizer.get_feature_names_out()
w = model.coef_.ravel()

k = 30
top_fake_idx = np.argsort(w)[-k:][::-1]
top_real_idx = np.argsort(w)[:k]

df_fake = pd.DataFrame({"token": feat[top_fake_idx], "weight": w[top_fake_idx], "direction": "push_fake(1)"})
df_real = pd.DataFrame({"token": feat[top_real_idx], "weight": w[top_real_idx], "direction": "push_real(0)"})
out_df = pd.concat([df_fake, df_real], ignore_index=True)

out_path = os.path.join(TABLES_DIR, "tfidf_linearsvc_top_tokens.csv")
out_df.to_csv(out_path, index=False, encoding="utf-8-sig")
print("Saved:", out_path)
print(out_df.head(10))

def plot_bar(df, title, out_base):
    df2 = df.sort_values("weight", ascending=True)
    plt.figure(figsize=(9, 7))
    plt.barh(df2["token"].tolist(), df2["weight"].tolist())
    plt.title(title)
    plt.xlabel("LinearSVC coefficient (positive pushes Fake=1)")
    plt.tight_layout()
    png = out_base + ".png"
    pdf = out_base + ".pdf"
    plt.savefig(png, dpi=250, bbox_inches="tight")
    plt.savefig(pdf, bbox_inches="tight")
    plt.show()
    print("Saved:", png)
    print("Saved:", pdf)

plot_bar(df_fake, "Top tokens pushing Fake(1) — TFIDF LinearSVC", os.path.join(FIGURES_DIR, "tfidf_linearsvc_top_fake"))
plot_bar(df_real, "Top tokens pushing Real(0) — TFIDF LinearSVC", os.path.join(FIGURES_DIR, "tfidf_linearsvc_top_real"))
